In [1]:
# %pip install -q -U "vllm>=0.17.0"
%pip show vllm

Name: vllm
Version: 0.32.0
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiohttp, anthropic, apache-tvm-ffi, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, einops, fastapi, fastsafetensors, filelock, flashinfer-python, huggingface_hub, humming-kernels, ijson, instanttensor, jsonschema, lark, llguidance, lm-format-enforcer, mcp, mistral_common, model-hosting-container-standards, msgspec, ninja, numba, numpy, nvidia-cudnn-frontend, nvidia-cutlass-dsl, nvtx, openai, opencv-python-headless, opentelemetry-api, opentelemetry-exporter-otlp, opentelemetry-sdk, opentelemetry-semantic-conventions-ai, oss-harmony, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, PyNvVideoCodec, python-json-lo

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [3]:
import hashlib
import json
import re

import pandas as pd
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams

In [4]:
MODEL_ID = "Qwen/Qwen3.6-35B-A3B-FP8"
DATASET_ID = "pameydorke/arcanumsearch-cross-platform"

NUM_QUERIES = 5
BATCH_SIZE = 64
MAX_RETRIES = 2

MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.90
MAX_NUM_SEQS = 128
MAX_NUM_BATCHED_TOKENS = 8192

BANNED_WORDS = ["arcanum", "experience", "reward"]
BANNED_RE = re.compile(r"\b(" + "|".join(BANNED_WORDS) + r")\b", re.IGNORECASE)

In [11]:
ds = load_dataset(DATASET_ID, split="train")

records = []
for i, row in enumerate(ds):
    link = (row.get("LINK") or "").strip()
    title = (row.get("TITLE") or "").strip()
    text = (row.get("TEXT") or "").strip()

    # doc_id =  hashlib.sha1(link.encode("utf-8")).hexdigest()[:16]
    doc_id = hashlib.sha1(text[:200].encode("utf-8")).hexdigest()[:16]
    records.append(
        {
            "doc_id": doc_id,
            "link": link,
            "title": title,
            "text": text,
        }
    )

df_docs = pd.DataFrame(records).drop_duplicates("doc_id").reset_index(drop=True)
print("Documents:", len(df_docs))
print(df_docs.head(3).to_string())

Documents: 1153
             doc_id                                                          link                     title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [12]:
PROMPT_TEMPLATE = """You are generating search queries for a retrieval tool used by an AI agent. The agent will type a query when it needs the information in the document below.

Document title: {title}
Document text:
{text}

Write {num_queries} concise, accurate search queries that should retrieve this exact document.

Requirements:
1. Each query must be a natural-language search string of 1 to 14 words.
2. Include specific names, places, items, skills, quests, or concepts from the document so the query is unambiguous.
3. Cover different aspects of the document; do not paraphrase the same idea {num_queries} times.
4. Use normal capitalization and punctuation. Do NOT use quotation marks, boolean operators (AND/OR), site: filters, or markdown.
5. Do NOT mention "document", "article", "page", "text", or "wiki".
6. Never use the words "arcanum", "experience", or "reward" (case-insensitive).
7. Do not make every query just the document title.
8. Check spelling and use the exact names from the document.

The queries should look like something a competent human or agent would actually type, not like a summarization task.

Return ONLY a JSON array of exactly {num_queries} strings, for example:
["query one", "query two", "query three", "query four", "query five"]

Queries:"""


def render_prompt(rec: dict, strict: bool = False) -> str:
    user = PROMPT_TEMPLATE.format(
        title=rec["title"],
        text=rec["text"],
        num_queries=NUM_QUERIES,
    )
    if strict:
        user = (
            "IMPORTANT: Your previous response was rejected. "
            "Follow every rule exactly and return only the JSON array.\n\n" + user
        )
    return user

In [13]:
from vllm.sampling_params import StructuredOutputsParams

QUERY_SCHEMA = {
    "type": "array",
    "minItems": NUM_QUERIES,
    "maxItems": NUM_QUERIES,
    "items": {
        "type": "string",
        "minLength": 3,
        "maxLength": 200,
    },
}


def make_sampling_params(temperature: float = 0.6) -> SamplingParams:
    structured = StructuredOutputsParams(json=QUERY_SCHEMA)

    kwargs = dict(
        temperature=temperature,
        top_p=0.9,
        max_tokens=220,
        repetition_penalty=1.05,
        bad_words=[
            "arcanum", "Arcanum", "ARCANUM",
            "experience", "Experience",
            "reward", "Reward",
        ],
    )
    kwargs["structured_outputs"] = structured
    return SamplingParams(**kwargs)


sp_main = make_sampling_params(temperature=0.6)
sp_retry = make_sampling_params(temperature=0.3)

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    enable_prefix_caching=True,
    disable_log_stats=True,
    # Text-only generation from a multimodal checkpoint
    limit_mm_per_prompt={"image": 0, "video": 0},
)

tokenizer = llm.get_tokenizer()

(MainProcess pid=1726) INFO 10-21 07:48:08 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'max_num_batched_tokens': 8192, 'max_num_seqs': 128, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3.6-35B-A3B-FP8'}


config.json:   0%|          | 0.00/37.0k [00:00<?, ?B/s]

(MainProcess pid=1726) INFO 10-21 07:48:28 [model.py:733] Resolved architecture: Qwen3_5MoeForConditionalGeneration
(MainProcess pid=1726) INFO 10-21 07:48:28 [model.py:2152] Using max model len 4096


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

(MainProcess pid=1726) INFO 10-21 07:48:31 [model.py:951] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(MainProcess pid=1726) INFO 10-21 07:48:31 [model.py:1008] Disabled mm_prefix attention mode because multimodal inputs are configuration-disabled. Attention backends without mm_prefix support may now be selected.
(MainProcess pid=1726) INFO 10-21 07:48:31 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=1726) INFO 10-21 07:48:31 [config.py:688] Mamba cache mode is set to 'align' for Qwen3_5MoeForConditionalGeneration by default when prefix caching is enabled
(MainProcess pid=1726) INFO 10-21 07:48:31 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=1726) INFO 10-21 07:48:31 [compilation.py:317] Enabled custom fusions: norm_quant, act_q

In [22]:
def apply_chat_template(user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def parse_json_array(text: str):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
    return None


def clean_query(q: str) -> str:
    q = q.strip().strip('"').strip("'").strip()
    q = re.sub(r"\s+", " ", q)
    return q


def validate_queries(qs) -> tuple[bool, list[str] | str]:
    if qs == None:
        return False, "parse_error"

    if not isinstance(qs, list):
        return False, "not_list"

    if len(qs) != NUM_QUERIES:
        return False, f"not_{NUM_QUERIES}_queries"
    
    cleaned = []
    seen = set()

    for q in qs:
        if not isinstance(q, str):
            return False, "non_string"

        q = clean_query(q)
        words = q.split()

        if not (1 <= len(words) <= 14):
            return False, "bad_length"

        if BANNED_RE.search(q):
            return False, "banned_word"

        key = q.lower()
        if key in seen:
            return False, "duplicate"

        seen.add(key)
        cleaned.append(q)

    return True, cleaned


def generate_batch(batch, sp_main, sp_retry, max_retries=MAX_RETRIES):
    """
    Generate queries for a batch of records.
    Returns (successful_results: dict[doc_id -> list[str]], failed_records: list).
    """
    results = {}
    pending = list(batch)

    for attempt in range(max_retries + 1):
        if not pending:
            break

        sp = sp_main if attempt == 0 else sp_retry
        prompts = [apply_chat_template(render_prompt(r, strict=(attempt > 0))) for r in pending]

        outputs = llm.generate(prompts, sampling_params=sp)

        next_pending = []
        for rec, out in zip(pending, outputs):
            text = out.outputs[0].text
            qs = parse_json_array(text)

            ok, val = validate_queries(qs)

            if ok:
                results[rec["doc_id"]] = val
            else:
                next_pending.append(rec)

        pending = next_pending

    return results, pending

In [27]:
all_records = df_docs.to_dict("records")

results_by_id: dict[str, list[str]] = {}
failed_records: list[dict] = []

for start in tqdm(range(0, len(all_records), BATCH_SIZE), desc="Generating queries"):
    batch = all_records[start : start + BATCH_SIZE]

    results, failed_batch = generate_batch(batch, sp_main, sp_retry)

    results_by_id.update(results)
    failed_records.extend(failed_batch)

    if failed_batch:
        print(f"Batch {start}: {len(failed_batch)} document(s) failed validation")

print(f"Done. Successful: {len(results_by_id)} | Failed: {len(failed_records)}")

Generating queries:   0%|          | 0/19 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 64/64 [00:03<00:00, 16.97it/s, est. speed input: 10384.85 toks/s, output: 993.82 toks/s]


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 64/64 [00:03<00:00, 18.03it/s, est. speed input: 10608.80 toks/s, output: 1023.93 toks/s]


...

Processed prompts: 100%|██████████| 4/4 [00:00<00:00,  4.91it/s, est. speed input: 4090.92 toks/s, output: 318.85 toks/s]

Batch 1088: 2 document(s) failed validation


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s, est. speed input: 754.65 toks/s, output: 127.18 toks/s]

Done. Successful: 1147 | Failed: 6


In [31]:
failed_records

[{'doc_id': '1548f60c056ba97e',
  'link': 'https://arcanum.fandom.com/wiki/Terra-Arcanum.com',
  'title': 'Terra-Arcanum.com',
  'text': 'Terra-Arcanum.com. Terra-Arcanum.com. Terra-Arcanum.com: The World of Troika Games is a long-running fansite of Arcanum: Of Steamworks and Magick Obscura and other games by defunct developer Troika Games. Activities. Aside from gameplay-related discussions, the community has authored several major patching and modding projects for Arcanum and can provide advice and a certain degree of support when it comes to patching Arcanum for newer computers and operating systems. Useful links. Terra-Arcanum front page (currently undergoing overhaul) Terra-Arcanum discussion forums Terra-Arcanum discussion forum rules Terra-Arcanum downloads section'},
 {'doc_id': 'a7b59aed87f4a950',
  'link': 'https://reddit.com/r/arcanum/comments/1fjk0ht/recommended_mods_2024/',
  'title': 'Recommended Mods? (2024)',
  'text': "Original Post: Recommended Mods? (2024). Hey y'all

In [32]:
successful = [r for r in all_records if r["doc_id"] in results_by_id]

df = pd.DataFrame(successful)
df["queries"] = df["doc_id"].map(results_by_id)

df = df[df["queries"].apply(lambda q: isinstance(q, list) and len(q) == NUM_QUERIES)].reset_index(drop=True)
print("Documents with 5 queries:", len(df))

pairs_df = (
    df[["doc_id", "link", "title", "text", "queries"]]
    .explode("queries")
    .rename(columns={"queries": "query"})
    .reset_index(drop=True)
)
pairs_ds = Dataset.from_pandas(pairs_df, preserve_index=False)

print(pairs_ds)

Documents with 5 queries: 1147
Dataset({
    features: ['doc_id', 'link', 'title', 'text', 'query'],
    num_rows: 5735
})


In [34]:
print("Query-count distribution:")
print(df["queries"].apply(len).value_counts().to_string())

all_queries = pairs_df["query"]
print("\nBanned-word matches:", int(all_queries.str.contains(BANNED_RE, regex=True).sum()))
print("Duplicate query strings:", int(all_queries.duplicated().sum()))
print("Duplicate query rate:", round(float(all_queries.duplicated().mean()), 4))

lengths = all_queries.str.split().str.len()
print("\nWord-count stats:")
print(lengths.describe().to_string())

print("\nRandom samples:")
for _, row in pairs_df.sample(min(15, len(pairs_df)), random_state=42).iterrows():
    print(f"- TITLE: {row['title']}")
    print(f"  QUERY: {row['query']}")

Query-count distribution:
queries
5    1147

Banned-word matches: 0
Duplicate query strings: 9
Duplicate query rate: 0.0016

Word-count stats:
count    5735.000000
mean        6.600697
std         1.365375
min         2.000000
25%         6.000000
50%         6.000000
75%         7.000000
max        14.000000

Random samples:
- TITLE: Dark Elf Camp
  QUERY: 345W 650S Dark Elf Camp coordinates
- TITLE: Why am I seeing double while scrolling? Anyone else have this bug? How can I fix this?
  QUERY: seeing double while scrolling Windows 10 Multiverse
- TITLE: Trying to run Arcanum on my macbook but wont work with unoffical patch.
  QUERY: same problem arcunum unofficial patch haven't found solution
- TITLE: Cynthia Boggs
  QUERY: Lorria nomadic women leader Cynthia escape
- TITLE: Is there a way to disable people moving while standing still?
  QUERY: stop character leg pumping animation bug
- TITLE: Mastery of the Magickal Arts
  QUERY: shaping the runes of magic personal development trans

/tmp/ipykernel_1726/2608533778.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("\nBanned-word matches:", int(all_queries.str.contains(BANNED_RE, regex=True).sum()))


In [39]:
import numpy as np
from datasets import Dataset, DatasetDict

SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1

# ---- 1. Split unique doc_ids ----
unique_docs = pairs_df["doc_id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_docs)

n = len(unique_docs)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

train_ids = set(unique_docs[:n_train])
val_ids   = set(unique_docs[n_train : n_train + n_val])
test_ids  = set(unique_docs[n_train + n_val :])

# ---- 2. Build train first, collect its query set ----
train_df = pairs_df[pairs_df["doc_id"].isin(train_ids)].reset_index(drop=True)
train_q  = set(train_df["query"].str.lower())

# ---- 3. Val: drop pairs whose query already appeared in train ----
val_df = pairs_df[pairs_df["doc_id"].isin(val_ids)].copy()
val_df = val_df[~val_df["query"].str.lower().isin(train_q)].reset_index(drop=True)
val_q  = set(val_df["query"].str.lower())

# ---- 4. Test: drop pairs seen in train OR val ----
test_df = pairs_df[pairs_df["doc_id"].isin(test_ids)].copy()
test_df = test_df[~test_df["query"].str.lower().isin(train_q | val_q)].reset_index(drop=True)
test_q  = set(test_df["query"].str.lower())

# ---- 5. Leakage assertions (both doc and query level) ----
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert not (train_q & val_q),  f"query overlap train∩val: {len(train_q & val_q)}"
assert not (train_q & test_q), f"query overlap train∩test: {len(train_q & test_q)}"
assert not (val_q & test_q),   f"query overlap val∩test: {len(val_q & test_q)}"

# ---- 6. Report what was dropped ----
val_before = len(pairs_df[pairs_df["doc_id"].isin(val_ids)])
test_before = len(pairs_df[pairs_df["doc_id"].isin(test_ids)])

print(f"Dropped overlapping pairs — val: {val_before - len(val_df)} | test: {test_before - len(test_df)}")
print(f"Val docs with ≥1 query: {val_df['doc_id'].nunique()}/{len(val_ids)}")
print(f"Test docs with ≥1 query: {test_df['doc_id'].nunique()}/{len(test_ids)}")
print(f"Docs — train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")
print(f"Pairs — train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

# ---- 7. Wrap in HF Datasets ----
pairs_splits = DatasetDict({
    "train":      Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df,   preserve_index=False),
    "test":       Dataset.from_pandas(test_df,  preserve_index=False),
})
print(pairs_splits)

Dropped overlapping pairs — val: 3 | test: 1
Val docs with ≥1 query: 114/114
Test docs with ≥1 query: 116/116
Docs — train: 917 | val: 114 | test: 116
Pairs — train: 4585 | val: 567 | test: 579
DatasetDict({
    train: Dataset({
        features: ['doc_id', 'link', 'title', 'text', 'query'],
        num_rows: 4585
    })
    validation: Dataset({
        features: ['doc_id', 'link', 'title', 'text', 'query'],
        num_rows: 567
    })
    test: Dataset({
        features: ['doc_id', 'link', 'title', 'text', 'query'],
        num_rows: 579
    })
})


In [ ]:
pairs_splits.push_to_hub("pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpw0kwczyr.parquet    : 100%|##########|  126kB /  126kB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8/commit/c963b05d9a29fcdfc89c09ca09e8227da6cc6310', commit_message='Upload dataset', commit_description='', oid='c963b05d9a29fcdfc89c09ca09e8227da6cc6310', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8'), pr_revision=None, pr_num=None)

(MainProcess pid=1726) INFO 10-21 08:50:06 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore
(MainProcess pid=1726) WARNING 10-21 08:50:06 [core_client.py:817] [shutdown] MPClient: engine core exited unexpectedly; starting cleanup
(MainProcess pid=1726) INFO 10-21 08:50:06 [core_client.py:770] [shutdown] MPClient: start timeout=default
(MainProcess pid=1726) INFO 10-21 08:50:06 [core_client.py:772] [shutdown] MPClient: stopping engine manager
(MainProcess pid=1726) INFO 10-21 08:50:06 [core_client.py:774] [shutdown] MPClient: engine manager stopped
(MainProcess pid=1726) INFO 10-21 08:50:06 [core_client.py:775] [shutdown] MPClient: cleaning up background resources
(MainProcess pid=1726) INFO 10-21 08:50:06 [core_client.py:777] [shutdown] MPClient: complete


In [2]:
import datasets

ds = datasets.load_dataset("pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8")

README.md:   0%|          | 0.00/622 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  911kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  114kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  126kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4585 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/567 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/579 [00:00<?, ? examples/s]

In [ ]:
shuffled_ds = ds.shuffle(seed=42)

shuffled_ds["train"][0]

{'doc_id': '7a0f63874a79697e',
 'link': 'https://arcanum.fandom.com/wiki/Pre-Generated_Characters',
 'title': 'Pre-Generated Characters',
 'text': 'Pre-Generated Characters. Pre-Generated Characters. "For those lacking the patience to endure the long and complex process of creating a Character of one’s own, the makers of this fine Game have provided a means of escaping this unwelcome tedium. Yes, gentle Player—you can begin your sojourn in Arcanum with all haste! For your personal convenience, a variety of pre-fabricated Characters have been made available, thus avoiding the muss and fuss of making one from scratch!" -Arcanum Players Manual. Intro. The very first thing that must be done before the adventure can begin is choosing the character YOU will play. The fundamental nature of a character is outlined by its race and gender. There are eight possible races, but for some races it is not possible to play a female. For those not wishing to Make a Custom Character, there is a possible 

In [4]:
shuffled_ds.push_to_hub("pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpcqot2fzq.parquet    : 100%|##########|  130kB /  130kB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8/commit/67d431d27f98fc3b0209ca4fd5c1499f6956df23', commit_message='Upload dataset', commit_description='', oid='67d431d27f98fc3b0209ca4fd5c1499f6956df23', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pameydorke/arcanumchat-cross-platform-qwen-36-a3b-fp8'), pr_revision=None, pr_num=None)